# Chunking

In [1]:
import torch
import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from lancedb.rerankers import ColbertReranker
import ollama
import os
import json
from tqdm.notebook import tqdm
import re, unicodedata


def clean_docling_chunk_strings(chunks):
    cleaned_chunks = []
    
    for chunk in chunks:
        # 2️⃣ Normalize Unicode and replace problematic punctuation
        chunk = unicodedata.normalize("NFKD", chunk).replace("\u00A0", " ")
        chunk = chunk.translate(str.maketrans({
            "–": "-", "—": "-", "‘": "'", "’": "'", "“": '"', "”": '"'
        }))

        # 3️⃣ Remove URLs (massive tokenizers killers)
        chunk = re.sub(r"http\S+", "", chunk)

        # 4️⃣ Normalize whitespace but preserve paragraphs
        chunk = re.sub(r"[ \t]+", " ", chunk)
        chunk = re.sub(r"\n\s*\n", "\n\n", chunk)  # merge single newlines, keep double
        chunk = chunk.strip()

        cleaned_chunks.append(chunk)

    return cleaned_chunks



EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
OLLAMA_MODEL_NAME= "anthropic_chunking"
# CHUNKS_WITH_METADATA_FILE_NAME = "preprocessed_chunks/anthropic_control_chunks_with_metadata.json"
CHUNKS_WITH_METADATA_FILE_NAME = "preprocessed_chunks/tobacco_traditional.json"
INPUT_DIR ="split_documents/smoking"
TABLE_NAME = "tobacco_traditional_table"

study_names = [f for f in os.listdir(INPUT_DIR) if f.endswith('.json')]
processed_chunks=[]
try:
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        processed_chunks = json.load(f)
except FileNotFoundError:
    print(f"No existing {CHUNKS_WITH_METADATA_FILE_NAME} file found, starting fresh.")
    

chunks_with_metadata = processed_chunks.copy()
processed_studies = set(chunk["document"] for chunk in processed_chunks)

study_names = [f for f in study_names if f not in processed_studies]
print(f"Found {len(processed_studies)} studies which are already processed.\nStudies which STILL need to be processed: {len(study_names)}:\n{study_names}...")


/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in ColPaliEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in SigLipEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


No existing preprocessed_chunks/tobacco_traditional.json file found, starting fresh.
Found 0 studies which are already processed.
Studies which STILL need to be processed: 4:
['CELEX_52016DC0269_EN_TXT.pdf.json', 'CELEX_52021DC0249_EN_TXT.pdf.json', 'methodology_technical-assessment_test-products_en.pdf.json', 'smoke-free_implementation_report_en.pdf.json']...


# Creating chunks and adding Metadata

As well as semantic context with ollama (Anthropic style)

In [ ]:
from codecarbon import EmissionsTracker

tracker_proposed = EmissionsTracker(
        project_name="tobacco_traditional",
        measure_power_secs=1,
        output_dir="./emissions_data"
    )


with tracker_proposed:
	for source in tqdm(study_names, desc="Chunking documents..."):   
		with open(f"{INPUT_DIR}/{source}", "r", encoding="utf-8") as f:
			chunks = json.load(f)
		chunks_str = [chunk["text"] for chunk in chunks]
		chunks_str = clean_docling_chunk_strings(chunks_str)
		entire_doc = " ".join(chunks_str)

		for chunk in tqdm(chunks, desc=f"Adding context for chunks of {source[:20]}...", leave=False):    
			chunk_index = chunks.index(chunk)

			entire_doc = "FULL DOCUMENT:\n" + entire_doc
			ollama_prompt = f"CHUNK:\n{chunks_str[chunk_index]}"
			history =  [{'role': 'user', 'content': entire_doc}, {'role': 'user', 'content': ollama_prompt}]

			response = ollama.chat(
				model=OLLAMA_MODEL_NAME,
				messages=history,
				options={
					"num_ctx": 30_000
				}
			)
			context = response['message']['content']
			text_to_embed = context + "\n\n" + chunks_str[chunk_index] 

			chunks_with_metadata.append({'text': text_to_embed, 'original_text':chunks_str[chunk_index], 'context':context, 'document':chunk['document'], 'id': chunk['id']})
			
	# Total runtime: 71m 34s for 25 documents

[codecarbon WARNING @ 16:29:56] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 16:29:56] [setup] RAM Tracking...
[codecarbon INFO @ 16:29:56] [setup] CPU Tracking...
[codecarbon WARNING @ 16:29:56] 	RAPL - Permission denied reading RAPL file /sys/class/powercap/intel-rapl/subsystem/intel-rapl-mmio/intel-rapl-mmio:0/energy_uj. You can grant read permission with: sudo chmod -R a+r /sys/class/powercap/*
[codecarbon WARNING @ 16:29:57] We saw that you have a Intel(R) Core(TM) i7-14650HX but we don't know it. Please contact us.
[codecarbon WARNING @ 16:29:57] We will use the default power consumption of 4 W per thread for your 24 CPU, so 96W.
[codecarbon WARNING @ 16:29:57] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 16:29:57] CPU Model on constant consumption mode: Int

Chunking documents...:   0%|          | 0/4 [00:00<?, ?it/s]

Adding context for chunks of CELEX_52016DC0269_EN...:   0%|          | 0/14 [00:00<?, ?it/s]

[codecarbon INFO @ 16:30:02] Energy consumed for RAM : 0.000011 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 16:30:03] Delta energy consumed for CPU with cpu_load : 0.000005 kWh, power : 9.6717624576 W
[codecarbon INFO @ 16:30:03] Energy consumed for All CPU : 0.000005 kWh
[codecarbon INFO @ 16:30:03] Energy consumed for all GPUs : 0.000012 kWh. Total GPU Power : 16.522358043034057 W
[codecarbon INFO @ 16:30:03] 0.000028 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 16:30:03] Energy consumed for RAM : 0.000014 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 16:30:04] Delta energy consumed for CPU with cpu_load : 0.000001 kWh, power : 9.703311936 W
[codecarbon INFO @ 16:30:04] Energy consumed for All CPU : 0.000007 kWh
[codecarbon INFO @ 16:30:04] Energy consumed for all GPUs : 0.000019 kWh. Total GPU Power : 28.109513368225503 W
[codecarbon INFO @ 16:30:04] 0.000040 kWh of electricity and 0.000000 L of water were used since the beginning.
[codeca

Adding context for chunks of CELEX_52021DC0249_EN...:   0%|          | 0/32 [00:00<?, ?it/s]

[codecarbon INFO @ 16:30:39] Delta energy consumed for CPU with cpu_load : 0.000001 kWh, power : 9.6393270336 W
[codecarbon INFO @ 16:30:39] Energy consumed for All CPU : 0.000053 kWh
[codecarbon INFO @ 16:30:39] Energy consumed for all GPUs : 0.000622 kWh. Total GPU Power : 65.47629845913565 W
[codecarbon INFO @ 16:30:39] 0.000784 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 16:30:39] Energy consumed for RAM : 0.000112 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 16:30:40] Delta energy consumed for CPU with cpu_load : 0.000001 kWh, power : 9.6397858608 W
[codecarbon INFO @ 16:30:40] Energy consumed for All CPU : 0.000054 kWh
[codecarbon INFO @ 16:30:40] Energy consumed for all GPUs : 0.000641 kWh. Total GPU Power : 69.63510124219128 W
[codecarbon INFO @ 16:30:40] 0.000807 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 16:30:40] Energy consumed for RAM : 0.000114 kWh. RAM Power : 20.0 W
[codecar

Adding context for chunks of methodology_technica...:   0%|          | 0/60 [00:00<?, ?it/s]

[codecarbon INFO @ 16:33:04] Delta energy consumed for CPU with cpu_load : 0.000001 kWh, power : 9.646827014400003 W
[codecarbon INFO @ 16:33:04] Energy consumed for All CPU : 0.000243 kWh
[codecarbon INFO @ 16:33:04] Energy consumed for all GPUs : 0.003069 kWh. Total GPU Power : 56.86122268308008 W
[codecarbon INFO @ 16:33:04] 0.003815 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 16:33:04] Energy consumed for RAM : 0.000506 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 16:33:05] Delta energy consumed for CPU with cpu_load : 0.000001 kWh, power : 9.616000675200002 W
[codecarbon INFO @ 16:33:05] Energy consumed for All CPU : 0.000244 kWh
[codecarbon INFO @ 16:33:05] Energy consumed for all GPUs : 0.003087 kWh. Total GPU Power : 65.4931682597913 W
[codecarbon INFO @ 16:33:05] 0.003837 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 16:33:05] Energy consumed for RAM : 0.000509 kWh. RAM Power : 20.0 W

Adding context for chunks of smoke-free_implement...:   0%|          | 0/35 [00:00<?, ?it/s]

[codecarbon INFO @ 16:40:40] Energy consumed for RAM : 0.001748 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 16:40:41] Delta energy consumed for CPU with cpu_load : 0.000001 kWh, power : 9.674077200000001 W
[codecarbon INFO @ 16:40:41] Energy consumed for All CPU : 0.000844 kWh
[codecarbon INFO @ 16:40:41] Energy consumed for all GPUs : 0.010851 kWh. Total GPU Power : 41.91427591393431 W
[codecarbon INFO @ 16:40:41] 0.013443 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 16:40:41] Energy consumed for RAM : 0.001751 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 16:40:42] Delta energy consumed for CPU with cpu_load : 0.000001 kWh, power : 9.6454674384 W
[codecarbon INFO @ 16:40:42] Energy consumed for All CPU : 0.000845 kWh
[codecarbon INFO @ 16:40:42] Energy consumed for all GPUs : 0.010867 kWh. Total GPU Power : 58.75754505334926 W
[codecarbon INFO @ 16:40:42] 0.013463 kWh of electricity and 0.000000 L of water were used since the beginning.
[co

In [4]:
# Save the the processed chunks in case VectorDB upload goes wrong.
# Luckily since this is a notebook, if the chunking is interrupted, we can still save the partial results here.
# Append new chunks to the existing file if it exists, otherwise create it
if os.path.exists(CHUNKS_WITH_METADATA_FILE_NAME):
    print(f"Appending to existing {CHUNKS_WITH_METADATA_FILE_NAME} file.")
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        existing_data = json.load(f)
    # Avoid duplicate entries by id
    existing_ids = {chunk['id'] for chunk in existing_data}
    new_chunks = [chunk for chunk in chunks_with_metadata if chunk['id'] not in existing_ids]
    chunks_with_metadata = existing_data + new_chunks

with open(CHUNKS_WITH_METADATA_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(chunks_with_metadata, f, ensure_ascii=False, indent=2)

print(f"Results saved to {CHUNKS_WITH_METADATA_FILE_NAME}")

Results saved to preprocessed_chunks/tobacco_traditional.json


# Creating Database

In [5]:
registry = get_registry()
hf = registry.get("huggingface").create(name=EMBEDDING_MODEL_NAME, trust_remote_code=True, device="cuda" if torch.cuda.is_available() else "cpu")


# Define model
class MyDocument(LanceModel):
    text: str = hf.SourceField()
    vector: Vector(hf.ndims()) = hf.VectorField()
    original_text: str
    context: str
    document: str
    id: str  # Unique identifier for the chunk


db = lancedb.connect("./db")
db.create_table(TABLE_NAME, schema=MyDocument, mode="overwrite") # Uncomment this line when running this cell for the first time
table = db.open_table(TABLE_NAME)

# Upload in batches with progress bar
with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
    chunks_with_metadata = json.load(f)

batch_size = 100
for i in tqdm(range(0, len(chunks_with_metadata), batch_size), desc="Uploading chunks to VectorDB"):
    batch = chunks_with_metadata[i:i+batch_size]
    table.add(batch)

table.create_scalar_index("id", replace=True) # Index based on the chunk's id, used to manually prevent duplicates

reranker = ColbertReranker()
table.create_fts_index("text", replace=True) # Used by the reranker as well as the hybrid search's BM25 index
table.wait_for_index(["text_idx"])  # Wait for the indexing to finish

<All keys matched successfully>


Uploading chunks to VectorDB:   0%|          | 0/2 [00:00<?, ?it/s]

<All keys matched successfully>
<All keys matched successfully>


Loading ColBERTRanker model colbert-ir/colbertv2.0 (this message can be suppressed by setting verbose=0)
No device set
Using device cuda
No dtype set
Using dtype torch.float32
Loading model colbert-ir/colbertv2.0, this might take a while...
Linear Dim set to: 128 for downcasting


# Example query

In [6]:
prompt = "How was stock market data gathered?"
results = table.search(prompt, query_type="hybrid", vector_column_name="vector", fts_columns="text") \
            .rerank(reranker=reranker) \
            .limit(5) \
            .to_pandas()


results

<All keys matched successfully>


,text,vector,original_text,context,document,id,_relevance_score
0,The document outlines a sampling strategy for ...,"[-0.058579333, 0.30300298, -4.2890224, -0.9924...",A random sampling among strata approach was us...,The document outlines a sampling strategy for ...,methodology_technical-assessment_test-products...,f810a31ccd3d86db0af8650011a07426c3d787aaaf391b...,0.620113
1,Details regarding the EU’s traceability system...,"[0.24163637, 0.6465223, -3.8736026, -0.6131732...","By the end of 2020, the EU traceability system...",Details regarding the EU’s traceability system...,CELEX_52021DC0249_EN_TXT.pdf,1697d3bd940ba7c0275705ad1e894a3079996a93667dbd...,0.580078
2,Provides the framework for how IAP will use te...,"[0.23077306, -0.13157046, -3.6705682, -1.81493...",Where IAP considers input from the technical g...,Provides the framework for how IAP will use te...,methodology_technical-assessment_test-products...,1e7fb4ae0ace0a15659f5e3a0d626e9af0ef938cd6b921...,0.484024
3,Chemical analysis was conducted to identify vo...,"[0.8535359, -1.0594655, -3.4595351, -1.0828127...","Following the sensory assessment, a qualitativ...",Chemical analysis was conducted to identify vo...,methodology_technical-assessment_test-products...,2df8bad7bbd8ce65d4d95f6005bccb6ed39fdf2147ca80...,0.444545
4,This chunk outlines the report's purpose: to a...,"[0.4440033, -0.13975774, -3.5378692, -1.597981...",Report on the implementation of the Council Re...,This chunk outlines the report's purpose: to a...,smoke-free_implementation_report_en.pdf,690a2e1824c985fc7939a5b771f5e8a11b78c4868763ea...,0.429683


In [ ]:
results.iloc[0,0]

In [ ]:
table.stats()